In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sklearn as skt
import xgboost as xgb
import json
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
import default_risk.config as cfg
import os
import xgboost as xgb
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.xgboost
from default_risk.scripts.auxiliars_for_modeling import cast_object_into_categoricals
from default_risk.scripts.auxiliars_for_modeling import get_baseline_setup
from default_risk.scripts.auxiliars_for_modeling import prepare_columns

import gc



load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)



,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
#the setup (we encapsulated that in a function for keep it constante during the joining to analize purely the gains of each table)
cv , hiperparams = get_baseline_setup() 

In [ ]:

#lets try with the main table without any treatment in the data.
application_train_df = pd.read_csv(cfg.RAW_DATA_DIR / "application_train.csv")

#minimun preparations necessary to be able to train the model with application_train
Y= application_train_df["TARGET"]
X= application_train_df.drop(columns=["TARGET"])
X.drop(columns=["SK_ID_CURR"],inplace=True)
X= cast_object_into_categoricals(X)



run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"baseline")
#0.744 OOF auc is our baseline.

#cleaning memory
del application_train_df,X,Y  
gc.collect()

[0]	validation_0-auc:0.71236
[1]	validation_0-auc:0.71930
[2]	validation_0-auc:0.72405
[3]	validation_0-auc:0.72679
[4]	validation_0-auc:0.72947
[5]	validation_0-auc:0.73310
[6]	validation_0-auc:0.73447
[7]	validation_0-auc:0.73675
[8]	validation_0-auc:0.73883
[9]	validation_0-auc:0.74064
[10]	validation_0-auc:0.74188
[11]	validation_0-auc:0.74247
[12]	validation_0-auc:0.74319
[13]	validation_0-auc:0.74389
[14]	validation_0-auc:0.74462
[15]	validation_0-auc:0.74511
[16]	validation_0-auc:0.74516
[17]	validation_0-auc:0.74618
[18]	validation_0-auc:0.74663
[19]	validation_0-auc:0.74675
[20]	validation_0-auc:0.74695
[21]	validation_0-auc:0.74678
[22]	validation_0-auc:0.74700
[23]	validation_0-auc:0.74782
[24]	validation_0-auc:0.74786
[25]	validation_0-auc:0.74823
[26]	validation_0-auc:0.74809
[27]	validation_0-auc:0.74793
[28]	validation_0-auc:0.74781
[29]	validation_0-auc:0.74841
[30]	validation_0-auc:0.74825
[31]	validation_0-auc:0.74841
[32]	validation_0-auc:0.74848
[33]	validation_0-au

420

In [ ]:
#now let's repeat the set up with the version of the silver layer (Cleaned application_train)
#Therefore, this part need execute make_dataset first to generate the fold 01_cleaned

cleaned_application_train= pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
X,Y = prepare_columns(cleaned_application_train)
X= cast_object_into_categoricals(X)


run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"baseline_cleaned_data")
#0.745 OOF auc. Just cleaning the data give us +0.1%, and with less risk of overfitting.


#cleaning memory
del cleaned_application_train,X,Y  
gc.collect()

[0]	validation_0-auc:0.71369
[1]	validation_0-auc:0.72214
[2]	validation_0-auc:0.72685
[3]	validation_0-auc:0.72993
[4]	validation_0-auc:0.73181
[5]	validation_0-auc:0.73344
[6]	validation_0-auc:0.73501
[7]	validation_0-auc:0.73769
[8]	validation_0-auc:0.74001
[9]	validation_0-auc:0.74191
[10]	validation_0-auc:0.74342
[11]	validation_0-auc:0.74406
[12]	validation_0-auc:0.74648
[13]	validation_0-auc:0.74719
[14]	validation_0-auc:0.74779
[15]	validation_0-auc:0.74797
[16]	validation_0-auc:0.74837
[17]	validation_0-auc:0.74823
[18]	validation_0-auc:0.74931
[19]	validation_0-auc:0.74960
[20]	validation_0-auc:0.74982
[21]	validation_0-auc:0.75001
[22]	validation_0-auc:0.75034
[23]	validation_0-auc:0.75034
[24]	validation_0-auc:0.75018
[25]	validation_0-auc:0.74978
[26]	validation_0-auc:0.74999
[27]	validation_0-auc:0.74972
[28]	validation_0-auc:0.74942
[29]	validation_0-auc:0.74937
[30]	validation_0-auc:0.74922
[31]	validation_0-auc:0.74987
[32]	validation_0-auc:0.74978
[33]	validation_0-au

211